# Fruit Detection Menggunakan YOLO11 + Roboflow Dataset

**Komputerisasi Industri Pertanian**

- **Nama:** ................................
- **NIM:** ................................
- **Kelas:** ................................
- **Dataset:** Roboflow Universe `tobeit68-ml/fruits-detection-flqbb`
- **Metode:**
  - Object detection menggunakan **Ultralytics YOLO11** berbasis PyTorch.
  - Dataset diunduh dari **Roboflow** dalam format YOLO.
  - Model dilatih dengan transfer learning dari bobot pretrained `yolo11n.pt`.

Notebook ini disusun untuk dijalankan di **Google Colab**. Sebelum menjalankan seluruh cell, aktifkan GPU melalui menu **Runtime → Change runtime type → Hardware accelerator → GPU** agar proses training dan inference lebih cepat.

## 1. Cek runtime Google Colab, Python, PyTorch, dan GPU

Cell ini digunakan untuk memastikan lingkungan kerja sudah siap sebelum proses training dimulai. Informasi yang dicek meliputi versi Python, versi PyTorch, ketersediaan CUDA, serta nama GPU yang digunakan.


In [ ]:
# Cell ini mengecek lingkungan eksekusi Google Colab, versi Python, PyTorch, dan status GPU.

import os
import sys
import platform

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA tersedia:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
    else:
        print("PERINGATAN: GPU tidak terdeteksi. Aktifkan GPU melalui Runtime > Change runtime type > GPU.")
except Exception as e:
    print("PyTorch belum terpasang atau gagal diimpor:", e)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

## 2. Setup library utama

Cell ini menginstal library yang dibutuhkan untuk seluruh pipeline fruit detection. Library utama yang digunakan adalah `ultralytics` untuk YOLO11, `roboflow` untuk mengambil dataset dari Roboflow Universe, `opencv-python-headless` untuk pengolahan citra/video, serta library pendukung seperti `matplotlib`, `pandas`, `Pillow`, `pyyaml`, dan `tqdm`.

Instalasi biasanya hanya perlu dilakukan sekali pada setiap runtime Colab baru. Setelah instalasi selesai, `ultralytics.checks()` akan menampilkan ringkasan kompatibilitas environment.

In [ ]:
# Cell ini menginstal pustaka utama yang dibutuhkan:
# - ultralytics: implementasi YOLO11 berbasis PyTorch
# - roboflow: SDK untuk mengunduh dataset dari Roboflow Universe
# - opencv-python-headless: membaca/menulis gambar dan video tanpa GUI desktop
# - supervision: utilitas tambahan untuk computer vision

!pip -q install --upgrade ultralytics roboflow supervision opencv-python-headless pillow matplotlib pandas pyyaml tqdm

import ultralytics
ultralytics.checks()

## 3. Import library dan konfigurasi eksperimen

Cell ini mengimpor semua library yang akan dipakai dan menetapkan konfigurasi utama eksperimen. Parameter penting yang dapat diatur meliputi workspace Roboflow, nama project dataset, versi dataset, ukuran citra input, jumlah epoch, batch size, confidence threshold, serta varian model YOLO11.

Untuk percobaan awal, `MODEL_VARIANT = "n"` digunakan karena model nano lebih ringan dan cocok untuk Google Colab. Jika GPU mencukupi dan ingin akurasi lebih tinggi, varian dapat diganti menjadi `s`, `m`, `l`, atau `x` tanpa mengubah alur notebook.

In [ ]:
# Cell ini mengimpor library dan mengatur parameter utama eksperimen.
# Jika ingin training lebih cepat, kurangi EPOCHS. Jika ingin akurasi lebih baik, naikkan EPOCHS.
# MODEL_VARIANT dapat diganti: 'n' = nano/cepat, 's' = small, 'm' = medium, 'l' = large, 'x' = extra large.

from pathlib import Path
import os
import random
import shutil
import yaml
import json
import glob
import time
import subprocess
from base64 import b64decode, b64encode
from getpass import getpass

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML, Javascript, Image as IPyImage

import torch
from ultralytics import YOLO

# -------------------------
# Konfigurasi dataset Roboflow
# -------------------------
WORKSPACE = "tobeit68-ml"
PROJECT = "fruits-detection-flqbb"
VERSION = 1

# Format ekspor Roboflow yang akan dicoba berurutan.
EXPORT_FORMATS = ["yolov11", "yolov8"]

# -------------------------
# Konfigurasi training YOLO11
# -------------------------
MODEL_VARIANT = "n"      # pilihan: n, s, m, l, x
IMG_SIZE = 640
EPOCHS = 50              # untuk uji cepat bisa ubah menjadi 10-20
BATCH = 16               # jika GPU memory error, ubah menjadi 8 atau 4
CONF_THRESHOLD = 0.25
SEED = 42

# Folder kerja di Colab
BASE_DIR = Path("/content/fruit_detection_yolo11")
DATASET_DIR = BASE_DIR / "datasets" / "fruits_detection"
RUNS_DIR = BASE_DIR / "runs"
BASE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device yang digunakan:", DEVICE)
print("Folder kerja:", BASE_DIR)

## 4. Mengambil Roboflow API Key

Dataset Roboflow membutuhkan API key agar notebook dapat mengunduh dataset secara otomatis. Cell ini terlebih dahulu mencoba membaca secret bernama `ROBOFLOW_API_KEY` dari fitur **Colab Secrets**. Jika belum tersedia, pengguna akan diminta memasukkan API key secara manual melalui input aman.

Agar lebih praktis, simpan API key di Colab dengan nama `ROBOFLOW_API_KEY`. Dengan cara ini, notebook dapat dijalankan ulang tanpa perlu mengetik ulang API key setiap kali runtime dimulai.

In [ ]:
# Cell ini mengambil Roboflow API Key dari Colab Secrets dengan nama ROBOFLOW_API_KEY.
# Jika secret belum tersedia, notebook akan meminta API key melalui input aman/getpass.
# Cara membuat API key: login Roboflow > Account Settings > API Keys.

ROBOFLOW_API_KEY = None

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Masukkan Roboflow API Key Anda: ")

if not ROBOFLOW_API_KEY:
    raise ValueError("Roboflow API Key belum diisi. Simpan di Colab Secrets atau masukkan manual saat diminta.")
else:
    print("API key berhasil dibaca.")

## 5. Download dataset dari Roboflow Universe

Cell ini mengunduh dataset `fruits-detection-flqbb` dari workspace `tobeit68-ml` berdasarkan versi dataset yang sudah ditentukan. Format ekspor `yolov11` dicoba terlebih dahulu karena notebook menggunakan YOLO11. Jika format tersebut gagal, notebook akan mencoba format `yolov8` sebagai fallback karena struktur YAML YOLO Ultralytics masih kompatibel untuk proses training.

Hasil unduhan umumnya berisi file `data.yaml` serta folder `train`, `valid`, dan `test` yang masing-masing memuat subfolder `images` dan `labels`.

In [ ]:
# Cell ini mengunduh dataset Roboflow Universe berdasarkan workspace, project, dan version.
# Notebook mencoba format 'yolov11' terlebih dahulu. Jika gagal, notebook mencoba 'yolov8' sebagai fallback.
# Hasil unduhan biasanya berisi data.yaml, folder train/images, train/labels, valid/images, dan test/images.

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)

DATASET_DIR.mkdir(parents=True, exist_ok=True)

dataset = None
last_error = None
for fmt in EXPORT_FORMATS:
    try:
        print(f"Mencoba mengunduh dataset dalam format: {fmt}")
        dataset = version.download(fmt, location=str(DATASET_DIR), overwrite=True)
        print(f"Berhasil mengunduh dataset dengan format: {fmt}")
        break
    except Exception as e:
        last_error = e
        print(f"Gagal format {fmt}: {e}")

if dataset is None:
    raise RuntimeError(f"Semua format ekspor gagal. Error terakhir: {last_error}")

DATA_YAML = Path(dataset.location) / "data.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"data.yaml tidak ditemukan di {DATA_YAML}")

print("Lokasi dataset:", dataset.location)
print("File YAML:", DATA_YAML)

## 6. Membaca `data.yaml` dan daftar kelas dataset

Cell ini membaca isi file `data.yaml` dari dataset Roboflow. File ini penting karena berisi lokasi folder data, jumlah kelas, dan nama kelas yang akan dikenali oleh model.

Pada dataset ini, kelas buah yang digunakan adalah Apple, Orange, Banana, Mango, Pineapple, dan Watermelon. Pemeriksaan ini dilakukan sebelum training agar struktur dataset dan nama kelas dapat dipastikan benar.

In [ ]:
# Cell ini membaca data.yaml dan mengecek struktur folder dataset.
# Berdasarkan halaman Roboflow, salah satu versi dataset dapat memiliki train set saja.
# Jika folder valid/test kosong, cell berikutnya akan membuat split validasi dan test secara otomatis dari train set.

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Isi data.yaml:")
print(json.dumps(data_cfg, indent=2))

# Ambil daftar nama kelas, mendukung format list atau dict dari YAML.
names = data_cfg.get("names", [])
if isinstance(names, dict):
    class_names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    class_names = list(names)

print("Jumlah kelas:", len(class_names))
print("Nama kelas:", class_names)

## 7. Validasi struktur dataset dan pembuatan split train-valid-test

Cell ini berisi fungsi bantu untuk memeriksa jumlah gambar, mencari file label, memindahkan gambar beserta labelnya, dan memperbaiki file `data.yaml` agar sesuai dengan standar Ultralytics YOLO11.

Jika dataset yang diunduh hanya memiliki data train atau folder valid/test kosong, fungsi ini akan membuat split lokal secara otomatis. Dengan demikian, notebook tetap dapat melakukan training, validasi, dan pengujian tanpa perlu membagi dataset secara manual.

In [ ]:
# Cell ini mendefinisikan fungsi bantu untuk menghitung jumlah gambar, mencari folder gambar/label,
# dan memperbaiki data.yaml agar kompatibel dengan Ultralytics YOLO11.
# Jika validasi tidak ada, dataset akan dibagi ulang secara lokal: train, valid, dan test.

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(folder: Path):
    if not folder.exists():
        return []
    files = []
    for ext in IMAGE_EXTS:
        files.extend(folder.glob(f"*{ext}"))
        files.extend(folder.glob(f"*{ext.upper()}"))
    return sorted(files)


def count_images(folder: Path):
    return len(list_images(folder))


def move_image_and_label(img_path: Path, src_img_dir: Path, src_lbl_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path):
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
    shutil.move(str(img_path), str(dst_img_dir / img_path.name))
    if lbl_path.exists():
        shutil.move(str(lbl_path), str(dst_lbl_dir / lbl_path.name))
    else:
        # Tetap buat label kosong agar struktur YOLO konsisten jika ada gambar tanpa objek.
        (dst_lbl_dir / f"{img_path.stem}.txt").write_text("")


def ensure_yolo_splits_and_yaml(data_yaml: Path, val_ratio=0.20, test_ratio=0.10, seed=42):
    """Memastikan dataset punya train/valid/test yang dapat dibaca Ultralytics."""
    root = data_yaml.parent
    train_img_dir = root / "train" / "images"
    train_lbl_dir = root / "train" / "labels"
    valid_img_dir = root / "valid" / "images"
    valid_lbl_dir = root / "valid" / "labels"
    test_img_dir = root / "test" / "images"
    test_lbl_dir = root / "test" / "labels"

    n_train_before = count_images(train_img_dir)
    n_valid_before = count_images(valid_img_dir)
    n_test_before = count_images(test_img_dir)

    print("Jumlah gambar sebelum perbaikan split:")
    print({"train": n_train_before, "valid": n_valid_before, "test": n_test_before})

    if n_train_before == 0:
        raise ValueError("Folder train/images kosong. Periksa hasil unduhan dataset Roboflow.")

    # Buat valid/test jika belum tersedia.
    if n_valid_before == 0 or n_test_before == 0:
        imgs = list_images(train_img_dir)
        random.Random(seed).shuffle(imgs)

        n_total = len(imgs)
        n_val = max(1, int(round(n_total * val_ratio))) if n_total >= 5 else 1
        n_test = max(1, int(round(n_total * test_ratio))) if n_total >= 10 else 0

        # Jangan sampai train habis.
        if n_val + n_test >= n_total:
            n_val = max(1, n_total // 5)
            n_test = 0

        val_imgs = imgs[:n_val] if n_valid_before == 0 else []
        test_imgs = imgs[n_val:n_val+n_test] if n_test_before == 0 else []

        for img_path in val_imgs:
            move_image_and_label(img_path, train_img_dir, train_lbl_dir, valid_img_dir, valid_lbl_dir)

        # Perbarui daftar gambar train setelah sebagian dipindah ke valid.
        remaining_imgs = list_images(train_img_dir)
        random.Random(seed + 1).shuffle(remaining_imgs)
        for img_path in remaining_imgs[:len(test_imgs)]:
            move_image_and_label(img_path, train_img_dir, train_lbl_dir, test_img_dir, test_lbl_dir)

    with open(data_yaml, "r") as f:
        cfg = yaml.safe_load(f)

    # Gunakan path absolut agar aman di Colab meskipun working directory berubah.
    cfg["path"] = str(root)
    cfg["train"] = "train/images"
    cfg["val"] = "valid/images"
    if count_images(test_img_dir) > 0:
        cfg["test"] = "test/images"

    # Pastikan nc sesuai jumlah names jika names tersedia.
    if "names" in cfg:
        if isinstance(cfg["names"], dict):
            cfg["nc"] = len(cfg["names"])
        elif isinstance(cfg["names"], list):
            cfg["nc"] = len(cfg["names"])

    with open(data_yaml, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    print("Jumlah gambar setelah perbaikan split:")
    print({
        "train": count_images(train_img_dir),
        "valid": count_images(valid_img_dir),
        "test": count_images(test_img_dir),
    })
    print("data.yaml sudah diperbarui:")
    print(yaml.safe_dump(cfg, sort_keys=False))
    return cfg


data_cfg = ensure_yolo_splits_and_yaml(DATA_YAML, val_ratio=0.20, test_ratio=0.10, seed=SEED)

## 8. Visualisasi contoh gambar dan anotasi YOLO

Sebelum training, dataset perlu dicek secara visual. Cell ini menampilkan beberapa contoh gambar dari folder train beserta bounding box anotasi YOLO.

Tujuannya adalah memastikan bahwa label objek berada pada posisi yang benar, nama kelas terbaca sesuai, dan tidak ada masalah dasar pada anotasi. Kesalahan anotasi pada tahap ini dapat memengaruhi kualitas model setelah training.

In [ ]:
# Cell ini menampilkan beberapa contoh gambar dari dataset beserta bounding box anotasi YOLO.
# Tujuannya untuk memverifikasi bahwa label dan kelas sudah terbaca dengan benar sebelum training.

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg.get("names", [])
if isinstance(names, dict):
    class_names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    class_names = list(names)

def draw_yolo_annotations(image_path: Path, label_path: Path, class_names):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id, xc, yc, bw, bh = parts[:5]
            cls_id = int(float(cls_id))
            xc, yc, bw, bh = map(float, (xc, yc, bw, bh))

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            cv2.putText(img, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    return img

train_images = list_images(Path(data_cfg["path"]) / data_cfg["train"])
random.Random(SEED).shuffle(train_images)
sample_images = train_images[:6]

plt.figure(figsize=(14, 9))
for i, img_path in enumerate(sample_images, start=1):
    label_path = img_path.parents[1] / "labels" / f"{img_path.stem}.txt"
    annotated = draw_yolo_annotations(img_path, label_path, class_names)
    plt.subplot(2, 3, i)
    plt.imshow(annotated)
    plt.title(img_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Load model YOLO11 pretrained

Cell ini membuat model YOLO11 dari bobot pretrained COCO. Bobot pretrained digunakan sebagai dasar transfer learning sehingga model tidak belajar dari nol.

Pada konfigurasi awal, notebook menggunakan `yolo11n.pt` karena ringan, cepat, dan sesuai untuk uji coba di Google Colab maupun kamera realtime. Model yang lebih besar dapat meningkatkan akurasi, tetapi membutuhkan waktu training dan memori GPU yang lebih besar.

In [ ]:
# Cell ini membuat model YOLO11 dari bobot pretrained COCO.
# Untuk Colab standar, yolo11n.pt adalah pilihan aman karena ringan dan cocok untuk eksperimen realtime.
# Jika butuh akurasi lebih tinggi dan GPU mencukupi, ubah MODEL_VARIANT menjadi 's' atau 'm'.

MODEL_NAME = f"yolo11{MODEL_VARIANT}.pt"
model = YOLO(MODEL_NAME)
print("Model yang digunakan:", MODEL_NAME)
print(model)

## 10. Training atau fine-tuning YOLO11 pada dataset buah

Cell ini menjalankan proses training YOLO11 menggunakan dataset buah dari Roboflow. Model akan mempelajari pola visual setiap kelas buah berdasarkan gambar dan bounding box yang tersedia.

Selama training, Ultralytics akan menyimpan hasil eksperimen ke folder `runs`, termasuk bobot `best.pt`, `last.pt`, grafik loss, metrik validasi, confusion matrix, dan kurva evaluasi. Bobot `best.pt` akan digunakan untuk tahap pengujian berikutnya.

In [ ]:
# Cell ini menjalankan training/fine-tuning YOLO11 pada dataset buah.
# Output training akan tersimpan di RUNS_DIR, termasuk bobot best.pt dan last.pt.
# Pada dataset kecil, gunakan augmentasi default Ultralytics dan pantau mAP, precision, recall, serta loss.

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="fruits_yolo11_train",
    exist_ok=True,
    patience=20,
    seed=SEED,
    plots=True,
    verbose=True,
)

BEST_MODEL_PATH = RUNS_DIR / "fruits_yolo11_train" / "weights" / "best.pt"
LAST_MODEL_PATH = RUNS_DIR / "fruits_yolo11_train" / "weights" / "last.pt"

if BEST_MODEL_PATH.exists():
    print("Bobot terbaik:", BEST_MODEL_PATH)
elif LAST_MODEL_PATH.exists():
    BEST_MODEL_PATH = LAST_MODEL_PATH
    print("best.pt belum tersedia, menggunakan last.pt:", BEST_MODEL_PATH)
else:
    raise FileNotFoundError("Bobot hasil training tidak ditemukan.")

## 11. Menampilkan grafik hasil training

Setelah training selesai, cell ini menampilkan visualisasi hasil yang dibuat otomatis oleh Ultralytics. Grafik `results.png` menunjukkan perubahan nilai loss dan metrik validasi dari epoch ke epoch.

Selain itu, confusion matrix dan PR curve juga ditampilkan jika tersedia. Visualisasi ini membantu mengevaluasi apakah model sudah belajar dengan baik atau masih perlu perbaikan dataset, parameter training, atau jumlah epoch.

In [ ]:
# Cell ini menampilkan grafik hasil training yang dibuat otomatis oleh Ultralytics.
# File results.png memperlihatkan perkembangan loss dan metrik validasi per epoch.

results_png = RUNS_DIR / "fruits_yolo11_train" / "results.png"
confusion_png = RUNS_DIR / "fruits_yolo11_train" / "confusion_matrix.png"
pr_curve_png = RUNS_DIR / "fruits_yolo11_train" / "PR_curve.png"

for p in [results_png, confusion_png, pr_curve_png]:
    if p.exists():
        print("Menampilkan:", p.name)
        display(IPyImage(filename=str(p)))
    else:
        print("File belum ditemukan:", p)

## 12. Validasi ulang menggunakan bobot terbaik

Cell ini memuat kembali model dari bobot terbaik, yaitu `best.pt`, lalu menjalankan validasi ulang pada dataset validasi. Metrik yang penting diperhatikan adalah precision, recall, mAP50, dan mAP50-95.

Validasi ulang berguna untuk memastikan performa model setelah training selesai. Jika hasil validasi rendah, beberapa perbaikan yang dapat dilakukan adalah menambah data, memperbaiki anotasi, menambah epoch, atau menggunakan varian model YOLO11 yang lebih besar.

In [ ]:
# Cell ini melakukan validasi ulang dengan bobot terbaik.
# Nilai yang perlu diperhatikan: mAP50, mAP50-95, precision, dan recall.
# Jika metrik rendah, pertimbangkan menambah data, memperbaiki anotasi, atau menaikkan epoch/model size.

trained_model = YOLO(str(BEST_MODEL_PATH))
val_metrics = trained_model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    split="val",
    plots=True,
)

print("Ringkasan metrics object:")
print(val_metrics)

## 13. Inference pada gambar validasi atau test dataset

Cell ini menguji model pada gambar dari folder validasi atau test. Model akan mendeteksi buah, menggambar bounding box, menampilkan nama kelas, dan memberikan confidence score.

Hasil prediksi disimpan ke folder eksperimen sehingga dapat digunakan untuk dokumentasi laporan. Cell ini juga menampilkan beberapa gambar hasil deteksi sebagai contoh output model setelah training.

In [ ]:
# Cell ini menguji model pada beberapa gambar validasi/test dan menyimpan hasil prediksi.
# Gambar hasil prediksi akan berisi bounding box, nama kelas buah, dan confidence score.

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

root = Path(data_cfg["path"])
valid_dir = root / data_cfg["val"]
test_dir = root / data_cfg.get("test", data_cfg["val"])
source_dir = test_dir if count_images(test_dir) > 0 else valid_dir

predict_results = trained_model.predict(
    source=str(source_dir),
    imgsz=IMG_SIZE,
    conf=CONF_THRESHOLD,
    save=True,
    project=str(RUNS_DIR),
    name="predict_dataset_samples",
    exist_ok=True,
    device=DEVICE,
)

pred_dir = RUNS_DIR / "predict_dataset_samples"
pred_images = list_images(pred_dir)
print("Folder hasil prediksi:", pred_dir)
print("Jumlah gambar hasil prediksi:", len(pred_images))

# Tampilkan maksimal 6 gambar hasil prediksi.
plt.figure(figsize=(14, 9))
for i, img_path in enumerate(pred_images[:6], start=1):
    img = Image.open(img_path).convert("RGB")
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 14. Uji coba pada gambar yang diunggah sendiri

Cell ini menyediakan pengujian model pada gambar eksternal yang diunggah langsung ke Colab. Pengguna dapat mengunggah gambar buah dalam format umum seperti `.jpg`, `.jpeg`, `.png`, atau `.webp`.

Setelah gambar diunggah, model akan menjalankan prediksi dan menampilkan hasil deteksi. Bagian ini berguna untuk melihat kemampuan generalisasi model pada gambar yang tidak berasal dari dataset training.

In [ ]:
# Cell ini menyediakan uji coba pada gambar yang Anda unggah sendiri.
# Jalankan cell, upload gambar buah, lalu model akan menampilkan hasil deteksi.
# Format umum yang aman digunakan: .jpg, .jpeg, .png, .webp.

from google.colab import files

uploaded = files.upload()
if len(uploaded) == 0:
    raise ValueError("Tidak ada gambar yang diunggah.")

uploaded_image_path = Path(next(iter(uploaded.keys()))).resolve()
print("Gambar uji:", uploaded_image_path)

single_results = trained_model.predict(
    source=str(uploaded_image_path),
    imgsz=IMG_SIZE,
    conf=CONF_THRESHOLD,
    save=True,
    project=str(RUNS_DIR),
    name="predict_uploaded_image",
    exist_ok=True,
    device=DEVICE,
)

out_dir = RUNS_DIR / "predict_uploaded_image"
out_images = list_images(out_dir)
if out_images:
    display(IPyImage(filename=str(out_images[-1])))
else:
    print("Hasil prediksi gambar belum ditemukan di", out_dir)

## 15. Uji coba deteksi buah pada video

Cell ini digunakan untuk menguji model pada video yang diunggah ke Google Colab. Model akan membaca setiap frame video, melakukan deteksi buah, lalu menyimpan video baru yang sudah diberi bounding box dan label kelas.

Agar hasil video mudah diputar di notebook, file output dikonversi ke format MP4 H.264 menggunakan `ffmpeg`. Format video input yang umum digunakan adalah `.mp4`, `.mov`, `.avi`, dan `.mkv`.

In [ ]:
# Cell ini menyediakan uji coba deteksi buah pada video yang diunggah ke Colab.
# Setelah video diproses, hasil akan dikonversi ke MP4 H.264 agar mudah diputar langsung di notebook.
# Format video yang umum digunakan: .mp4, .mov, .avi, .mkv.

from google.colab import files

uploaded = files.upload()
if len(uploaded) == 0:
    raise ValueError("Tidak ada video yang diunggah.")

video_path = Path(next(iter(uploaded.keys()))).resolve()
print("Video uji:", video_path)

video_results = trained_model.predict(
    source=str(video_path),
    imgsz=IMG_SIZE,
    conf=CONF_THRESHOLD,
    save=True,
    project=str(RUNS_DIR),
    name="predict_uploaded_video",
    exist_ok=True,
    device=DEVICE,
)

video_pred_dir = RUNS_DIR / "predict_uploaded_video"
video_outputs = []
for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
    video_outputs.extend(video_pred_dir.glob(ext))

if not video_outputs:
    raise FileNotFoundError(f"Video hasil prediksi tidak ditemukan di {video_pred_dir}")

raw_output_video = sorted(video_outputs, key=lambda p: p.stat().st_mtime)[-1]
preview_video = BASE_DIR / "fruits_detection_video_preview.mp4"

# Konversi agar browser Colab bisa memutar video dengan baik.
cmd = [
    "ffmpeg", "-y", "-i", str(raw_output_video),
    "-vcodec", "libx264", "-pix_fmt", "yuv420p",
    "-acodec", "aac", "-movflags", "+faststart",
    str(preview_video)
]
subprocess.run(cmd, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("Video hasil prediksi:", preview_video)

mp4 = open(preview_video, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width="720" controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")

## 16. Utility kamera realtime di Google Colab

Google Colab tidak mengakses webcam secara langsung seperti aplikasi desktop. Karena itu, cell ini menyiapkan fungsi JavaScript untuk membuka kamera browser, mengambil frame, dan mengirimkannya ke Python.

Frame dari kamera dikonversi menjadi citra OpenCV, diproses oleh YOLO11, lalu hasil deteksi dikirim kembali ke browser untuk ditampilkan. Proses ini memungkinkan simulasi deteksi realtime meskipun tetap memiliki latensi karena komunikasi JavaScript–Python.

In [ ]:
# Cell ini menyiapkan fungsi bantu untuk kamera realtime di Google Colab.
# Karena Colab berjalan di browser, akses kamera dilakukan melalui JavaScript, lalu frame dikirim ke Python untuk diprediksi YOLO11.
# Deteksi realtime di Colab memiliki latensi; untuk realtime produksi lebih stabil gunakan laptop lokal atau edge device.

from google.colab.output import eval_js


def js_to_bgr_image(js_reply: str):
    """Mengubah data URL dari JavaScript menjadi citra BGR OpenCV."""
    image_bytes = b64decode(js_reply.split(",")[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=cv2.IMREAD_COLOR)
    return img


def bgr_image_to_js(img_bgr: np.ndarray, quality=85):
    """Mengubah citra BGR OpenCV menjadi data URL JPEG untuk ditampilkan di browser."""
    ok, buffer = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        raise ValueError("Gagal encode frame ke JPEG.")
    return "data:image/jpeg;base64," + b64encode(buffer).decode("utf-8")


def setup_colab_camera(width=640, height=480):
    """Membuka kamera browser dan menyiapkan fungsi captureFrame/showPrediction di JavaScript."""
    display(Javascript(f"""
    async function setupCamera() {{
      const old = document.getElementById('fruit-yolo-camera-panel');
      if (old) old.remove();

      const panel = document.createElement('div');
      panel.id = 'fruit-yolo-camera-panel';
      panel.style = 'border:1px solid #ccc; padding:12px; margin:8px 0; width:{width + 40}px';
      panel.innerHTML = `
        <p><b>Kamera realtime YOLO11 Fruit Detection</b></p>
        <p>Izinkan akses kamera saat browser meminta permission. Klik tombol stop atau jalankan cell Stop Camera untuk mematikan kamera.</p>
        <video id="fruit-yolo-video" width="{width}" height="{height}" autoplay playsinline style="border:1px solid #ddd"></video>
        <p><b>Hasil deteksi:</b></p>
        <img id="fruit-yolo-prediction" width="{width}" style="border:1px solid #ddd" />
        <br><button onclick="window.stopFruitYoloCamera()">Stop Camera</button>
      `;
      document.body.appendChild(panel);

      const video = document.getElementById('fruit-yolo-video');
      const stream = await navigator.mediaDevices.getUserMedia({{video: {{width: {width}, height: {height}}}, audio: false}});
      window.fruitYoloStream = stream;
      video.srcObject = stream;
      await video.play();

      window.captureFruitYoloFrame = async function() {{
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth || {width};
        canvas.height = video.videoHeight || {height};
        const ctx = canvas.getContext('2d');
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        return canvas.toDataURL('image/jpeg', 0.85);
      }};

      window.showFruitYoloPrediction = function(imgData) {{
        const img = document.getElementById('fruit-yolo-prediction');
        img.src = imgData;
      }};

      window.stopFruitYoloCamera = function() {{
        if (window.fruitYoloStream) {{
          window.fruitYoloStream.getTracks().forEach(track => track.stop());
        }}
      }};

      return true;
    }}
    setupCamera();
    """))
    time.sleep(2)
    print("Kamera disiapkan. Jika belum muncul, pastikan browser memberi izin akses kamera.")

## 17. Inference kamera live di Google Colab

Cell ini menjalankan deteksi buah secara realtime dari kamera browser. Setiap frame kamera diambil, diproses oleh model YOLO11, lalu hasil bounding box ditampilkan kembali pada panel output notebook.

Parameter `MAX_FRAMES` menentukan jumlah frame yang diproses sebelum loop berhenti otomatis. Jika ingin menghentikan lebih cepat, gunakan tombol stop/interrupt pada cell atau jalankan cell stop camera setelahnya.

In [ ]:
# Cell ini menjalankan deteksi buah dari kamera realtime di Colab.
# Atur MAX_FRAMES untuk menentukan durasi perkiraan pemrosesan. Semakin besar nilainya, semakin lama loop berjalan.
# Untuk menghentikan lebih cepat, klik tombol stop/interrupt runtime pada cell ini.

MAX_FRAMES = 300       # contoh: 300 frame; ubah sesuai kebutuhan
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_CONF = CONF_THRESHOLD

setup_colab_camera(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)

camera_model = YOLO(str(BEST_MODEL_PATH))

for frame_idx in range(MAX_FRAMES):
    try:
        js_frame = eval_js("captureFruitYoloFrame()")
        frame_bgr = js_to_bgr_image(js_frame)

        results = camera_model.predict(
            source=frame_bgr,
            imgsz=IMG_SIZE,
            conf=CAMERA_CONF,
            device=DEVICE,
            verbose=False,
        )

        annotated_bgr = results[0].plot()
        out_js = bgr_image_to_js(annotated_bgr, quality=85)
        eval_js(f"showFruitYoloPrediction('{out_js}')")

        if frame_idx % 30 == 0:
            print(f"Frame diproses: {frame_idx}/{MAX_FRAMES}")
    except KeyboardInterrupt:
        print("Loop kamera dihentikan manual.")
        break
    except Exception as e:
        print("Terjadi error pada kamera realtime:", e)
        break

print("Loop kamera selesai. Jalankan cell Stop Camera untuk memastikan kamera mati.")

## 18. Stop kamera browser

Cell ini digunakan untuk mematikan stream kamera setelah pengujian realtime selesai agar kamera laptop tidak terus aktif setelah loop inference dihentikan.

Jalankan cell ini apabila lampu kamera masih menyala, panel kamera masih aktif, atau ketika ingin mengakhiri sesi uji realtime di Google Colab.

In [ ]:
# Cell ini mematikan kamera browser setelah uji realtime selesai.
# Jalankan cell ini jika lampu kamera masih menyala atau jika ingin menutup stream kamera.

try:
    eval_js("stopFruitYoloCamera()")
    print("Kamera dimatikan.")
except Exception as e:
    print("Tidak ada kamera aktif atau fungsi stop belum dibuat:", e)

## 19. Export model ke ONNX

Cell ini mengekspor model YOLO11 hasil training ke format ONNX. Format ONNX berguna jika model ingin digunakan di environment lain di luar Ultralytics, misalnya aplikasi desktop, server inference, atau runtime yang mendukung ONNX.

Untuk penggunaan langsung di Python dengan Ultralytics, file `best.pt` sudah cukup. Export ONNX bersifat opsional, tetapi bermanfaat untuk kebutuhan deployment lanjutan.

In [ ]:
# Cell ini mengekspor model ke format ONNX sebagai opsi deployment tambahan.
# ONNX berguna jika model ingin dipakai di environment lain, misalnya aplikasi desktop/server dengan runtime ONNX.

onnx_path = trained_model.export(format="onnx", imgsz=IMG_SIZE, dynamic=True, simplify=True)
print("Model ONNX tersimpan di:", onnx_path)

## 20. Membuat arsip ZIP hasil eksperimen

Cell ini membuat file ZIP yang berisi folder kerja notebook, hasil training, bobot model, grafik evaluasi, dan hasil prediksi. Arsip ini memudahkan pengguna mengunduh seluruh output eksperimen dari Google Colab.

File ZIP dapat diambil melalui panel **Files** di Colab. Jika ingin memunculkan dialog download otomatis, baris `files.download(archive_path)` dapat diaktifkan.

In [ ]:

# Cell ini membuat file ZIP berisi hasil training, bobot model, grafik evaluasi, dan hasil prediksi.
# Setelah ZIP dibuat, Anda dapat mengunduhnya dari panel Files Colab atau menggunakan files.download.

archive_base = BASE_DIR / "fruit_detection_yolo11_results"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(BASE_DIR))
print("Arsip hasil:", archive_path)

# Aktifkan baris berikut jika ingin langsung memunculkan dialog download.
from google.colab import files
files.download(archive_path)